In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
import lightgbm as lgb
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

In [31]:
# 한글 표시
plt.rcParams['font.family'] = "Gulim"
plt.rcParams['axes.unicode_minus'] = False

### 데이터 로드 및 기본 정보 확인

In [32]:
# 데이터 로드
df_merged = pd.read_csv('df_merged.csv')
y_targets = pd.read_csv('y_targets.csv')

In [33]:
# y_targets 컬럼 확인 및 명명
# 공모전 요구사항: 지상부/지하부 페놀, 플라보노이드 4개 타겟
target_names = ['Leaf_TPC', 'Root_TPC', 'Leaf_TFC', 'Root_TFC']  
y_targets = df_merged[target_names].copy()  

In [34]:
# 기본 정보
months = df_merged['month'].values
scenarios = df_merged['scenario'].values

In [35]:
print("\n데이터 구성:")
print(f"  총 샘플 수: {len(df_merged)}")
print(f"  특성 수: {df_merged.shape[1]}")
print(f"  타겟 수: {y_targets.shape[1]}")


데이터 구성:
  총 샘플 수: 405
  특성 수: 26
  타겟 수: 4


### 특정 그룹 정의 및 Feature Engineering

In [36]:
# 특성 그룹 정의
environmental = ['CO2ppm', 'Temp', 'Humid', 'VPD']
physiological = ['Chl_a', 'Chl_b', 'TChl', 'Car', 'Chl_a_b', 'TCh-Car']
fluorescence = ['ABS-RC', 'Dio-RC', 'Tro-RC', 'Eto-RC', 'PI_abs', 'DF_abs', 'SFI_abs', 'Fv-Fm']

In [37]:
# 원본 특성
original_features = environmental + physiological + fluorescence
X_original = df_merged[original_features].copy()

print(f"원본 특성 수: {X_original.shape[1]}")

원본 특성 수: 18


In [38]:
# 엔지니어링된 특성 추가
X_engineered = X_original.copy()

In [39]:
# 1. 생육 단계 (월별) - 천궁의 생장 패턴 반영
growth_map = {5: 1, 6: 2, 7: 3, 8: 4, 9: 5}
X_engineered['growth_stage'] = [growth_map[m] for m in months]
X_engineered['growth_stage_sq'] = X_engineered['growth_stage'] ** 2

In [40]:
# 2. 시나리오 정보 - 기후변화 시나리오별 특성
for scenario in ['SSP1', 'SSP3', 'SSP5']:
    X_engineered[f'is_{scenario}'] = (scenarios == scenario).astype(int)

In [41]:
# 3. 온도 스트레스 - 천궁 최적 온도(15-25°C) 기준
X_engineered['temp_stress'] = np.abs(X_engineered['Temp'] - 22)
X_engineered['temp_optimal'] = 1 / (1 + X_engineered['temp_stress'])

In [42]:
# 4. VPD 스트레스 - 수분 스트레스 지표
X_engineered['vpd_stress'] = np.where(X_engineered['VPD'] > 1.5, X_engineered['VPD'] - 1.5, 0)

In [43]:
# 5. CO2 효과 - 광합성 증진 효과
X_engineered['co2_effect'] = np.log1p(X_engineered['CO2ppm'] / 432)

In [44]:
# 6. 광합성 효율 지표
X_engineered['photosyn_efficiency'] = X_engineered['PI_abs'] * X_engineered['Fv-Fm']
X_engineered['energy_dissipation'] = X_engineered['Dio-RC'] / (X_engineered['ABS-RC'] + 1e-10)

In [45]:
# 7. 시나리오별 상호작용 특성
X_engineered['temp_x_ssp5'] = X_engineered['Temp'] * X_engineered['is_SSP5']
X_engineered['co2_x_ssp5'] = X_engineered['CO2ppm'] * X_engineered['is_SSP5']

In [46]:
# 8. 클로로필 비율 - 광합성 색소 균형
X_engineered['car_ratio'] = X_engineered['Car'] / (X_engineered['TChl'] + 1e-10)

### 상관관계 분석

In [47]:
# 전체 데이터에 대한 상관관계 분석
correlation_results = {}

for target in target_names:
    print(f"\n{target} 상관관계 분석:")
    correlations = {}
    
    # Pearson 상관계수 계산
    for feature in original_features:
        corr, p_value = pearsonr(X_original[feature], y_targets[target])
        if p_value < 0.05:  # 통계적으로 유의미한 상관관계만
            correlations[feature] = {
                'correlation': corr,
                'p_value': p_value,
                'abs_corr': abs(corr)
            }
    
    # 상관관계 강도순 정렬
    sorted_corr = sorted(correlations.items(), key=lambda x: x[1]['abs_corr'], reverse=True)
    
    print("  Top 5 상관관계 특성:")
    for feat, vals in sorted_corr[:5]:
        print(f"    {feat:20s}: r={vals['correlation']:+.3f} (p={vals['p_value']:.4f})")
    
    correlation_results[target] = sorted_corr


Leaf_TPC 상관관계 분석:
  Top 5 상관관계 특성:
    TCh-Car             : r=-0.441 (p=0.0000)
    Chl_a               : r=-0.353 (p=0.0000)
    SFI_abs             : r=+0.349 (p=0.0000)
    PI_abs              : r=+0.348 (p=0.0000)
    TChl                : r=-0.336 (p=0.0000)

Root_TPC 상관관계 분석:
  Top 5 상관관계 특성:
    Temp                : r=-0.659 (p=0.0000)
    VPD                 : r=-0.636 (p=0.0000)
    Car                 : r=+0.592 (p=0.0000)
    Humid               : r=+0.544 (p=0.0000)
    Tro-RC              : r=-0.455 (p=0.0000)

Leaf_TFC 상관관계 분석:
  Top 5 상관관계 특성:
    ABS-RC              : r=-0.653 (p=0.0000)
    Dio-RC              : r=-0.631 (p=0.0000)
    Tro-RC              : r=-0.610 (p=0.0000)
    DF_abs              : r=+0.588 (p=0.0000)
    Fv-Fm               : r=+0.585 (p=0.0000)

Root_TFC 상관관계 분석:
  Top 5 상관관계 특성:
    ABS-RC              : r=-0.580 (p=0.0000)
    Dio-RC              : r=-0.577 (p=0.0000)
    Car                 : r=+0.575 (p=0.0000)
    Fv-Fm               : r=

In [51]:
print("\n시나리오별 상관관계 차이 분석:")
scenario_correlations = {}

for scenario in ['SSP1', 'SSP3', 'SSP5']:
    scenario_mask = scenarios == scenario
    X_scenario = X_original[scenario_mask]
    y_scenario = y_targets[scenario_mask]
    
    scenario_correlations[scenario] = {}
    
    for target in target_names:  # target_names를 사용
        print(f"\n{target}:")
        correlations = {}
        for feature in original_features:
            corr, p_value = pearsonr(X_scenario[feature], y_scenario[target])
            if p_value < 0.05:
                correlations[feature] = {
                    'correlation': corr,
                    'p_value': p_value,
                    'abs_corr': abs(corr)
                }
        
        if correlations:
            sorted_corr = sorted(correlations.items(), key=lambda x: x[1]['abs_corr'], reverse=True)
            print(f"  Top 5 상관관계:")
            for feat, vals in sorted_corr[:5]:
                print(f"    {feat:20s}: r={vals['correlation']:+.3f}")
        
        scenario_correlations[scenario][target] = correlations


시나리오별 상관관계 차이 분석:

Leaf_TPC:
  Top 5 상관관계:
    PI_abs              : r=+0.887
    SFI_abs             : r=+0.886
    DF_abs              : r=+0.867
    ABS-RC              : r=-0.850
    Tro-RC              : r=-0.847

Root_TPC:
  Top 5 상관관계:
    Temp                : r=-0.852
    VPD                 : r=-0.831
    Humid               : r=+0.705
    Car                 : r=+0.573
    TChl                : r=+0.511

Leaf_TFC:
  Top 5 상관관계:
    PI_abs              : r=+0.822
    DF_abs              : r=+0.820
    SFI_abs             : r=+0.817
    Fv-Fm               : r=+0.814
    Dio-RC              : r=-0.806

Root_TFC:
  Top 5 상관관계:
    Humid               : r=+0.883
    Temp                : r=-0.848
    VPD                 : r=-0.843
    CO2ppm              : r=-0.710
    Car                 : r=+0.691

Leaf_TPC:
  Top 5 상관관계:
    Eto-RC              : r=+0.758
    Fv-Fm               : r=+0.589
    Dio-RC              : r=-0.551
    DF_abs              : r=+0.539
    Temp        

### 성능 평가 지표 정의

In [52]:
def calculate_mape(y_true, y_pred, eps=1e-9):
    """Mean Absolute Percentage Error"""
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

In [53]:
def calculate_pi(y_true, y_pred):
    """Performance Index (Nash-Sutcliffe Efficiency 기반)
    이유: 공모전에서 요구한 PI 지표 - 모델의 예측력을 0~1 범위로 평가"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / (ss_tot + 1e-10))

In [54]:
def calculate_comprehensive_metrics(y_true, y_pred):
    """공모전 요구 평가지표 모두 계산"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    # MSLE, RMSLE
    y_true_pos = np.maximum(y_true, 0)
    y_pred_pos = np.maximum(y_pred, 0)
    msle = mean_squared_error(np.log1p(y_true_pos), np.log1p(y_pred_pos))
    rmsle = np.sqrt(msle)
    
    r2 = r2_score(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    pi = calculate_pi(y_true, y_pred)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MSLE': msle,
        'RMSLE': rmsle,
        'R2': r2,
        'MAPE': mape,
        'PI': pi  # Performance Index 추가
    }

### 모델 정의 및 하이퍼파라미터 설정

In [55]:
models = {
    'LinearR': LinearRegression(),
    
    'Ridge': Ridge(alpha=1.0, random_state=42),
    
    'kNN': KNeighborsRegressor(
        n_neighbors=7,
        weights='distance',
        metric='minkowski',
        p=2
    ),
    
    'SVM': SVR(
        kernel='rbf',
        C=3.0,
        epsilon=0.05,
        gamma='scale'
    ),
    
    'DT': DecisionTreeRegressor(
        max_depth=15,
        min_samples_split=3,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42
    ),
    
    'RF': RandomForestRegressor(
        n_estimators=800,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ),
    
    'GB': GradientBoostingRegressor(
        n_estimators=400,
        max_depth=7,
        learning_rate=0.08,
        subsample=0.85,
        min_samples_split=3,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42
    ),
    
    'XGB': xgb.XGBRegressor(
        n_estimators=600,
        max_depth=9,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        colsample_bylevel=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_weight=2,
        gamma=0.1,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        verbose=0
    ),
    
    'LGBM': lgb.LGBMRegressor(
        n_estimators=500,
        max_depth=12,
        learning_rate=0.05,
        num_leaves=64,
        min_child_samples=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        min_split_gain=0.01,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

# 네이티브 다중출력 지원 모델
NATIVE_MULTIOUTPUT = ['RF', 'DT', 'kNN']

### 시나리오별 개별 모델링 with 교차검증

In [63]:
# 결과 저장
scenario_results = {}
cv_results = {}
best_models = {}

# 교차검증 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for scenario in ['SSP1', 'SSP3', 'SSP5']:
    print(f"\n=== {scenario} 시나리오 모델링 ===")
    
    # 해당 시나리오 데이터만 추출
    scenario_mask = scenarios == scenario
    X_scenario = X_engineered[scenario_mask]
    y_scenario = y_targets[scenario_mask]
    
    print(f"시나리오 데이터: {len(y_scenario)} samples")

    # 80/20 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X_scenario, y_scenario, test_size=0.2, random_state=42
    )
    
    print(f"Train: {len(y_train)}, Test: {len(y_test)}")
    
    # 스케일링
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    scenario_results[scenario] = {}
    cv_results[scenario] = {}
    
    best_mae = float('inf')
    best_model = None
    best_model_obj = None
    
    for model_name, base_model in models.items():
        from sklearn.base import clone
        model = clone(base_model)
        
        if model_name not in NATIVE_MULTIOUTPUT:
            model = MultiOutputRegressor(model)
        
        try:
            # 교차검증
            cv_scores = []
            for train_idx, val_idx in kf.split(X_train_scaled):
                X_cv_train, X_cv_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
                y_cv_train, y_cv_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
                
                model_cv = clone(model)
                model_cv.fit(X_cv_train, y_cv_train)
                pred_cv = model_cv.predict(X_cv_val)
                cv_mae = mean_absolute_error(y_cv_val, pred_cv)
                cv_scores.append(cv_mae)
            
            cv_results[scenario][model_name] = {
                'mean': np.mean(cv_scores),
                'std': np.std(cv_scores)
            }
            
            # 전체 훈련 데이터로 학습
            model.fit(X_train_scaled, y_train)
            pred = model.predict(X_test_scaled)
            
            # 평가
            metrics = calculate_comprehensive_metrics(y_test, pred)
            scenario_results[scenario][model_name] = metrics
            
            # 최고 성능 모델 추적
            if metrics['MAE'] < best_mae:
                best_mae = float(metrics['MAE'])  
                best_model = model_name
                best_model_obj = clone(model)
                best_model_obj.fit(X_train_scaled, y_train)
                
        except Exception as e:
            print(f"  {model_name} 오류: {e}")
    
    best_models[scenario] = {
        'model': best_model_obj,
        'scaler': scaler,
        'name': best_model
    }
    
    print(f"🏆 최고 성능: {best_model}")
    print(f"   MAE: {float(best_mae):.4f}")
    print(f"   R2: {float(scenario_results[scenario][best_model]['R2']):.4f}")
    print(f"   PI: {float(scenario_results[scenario][best_model]['PI']):.4f}")
    print(f"   CV MAE: {float(cv_results[scenario][best_model]['mean']):.4f} ± {float(cv_results[scenario][best_model]['std']):.4f}")


=== SSP1 시나리오 모델링 ===
시나리오 데이터: 135 samples
Train: 108, Test: 27
🏆 최고 성능: LGBM
   MAE: 0.0680
   R2: 0.9731


TypeError: cannot convert the series to <class 'float'>